#  5 · Gobernanza
Se crearan 3 tablas en un mismo esquema. Una para el World Port Index, una para almacenar la informacion estatica de los buques, y la del dataset completo AIS.


En una Free edition no se puede usar el parametro de Using Location cuando se define una tabla, por lo tanto las tablas creadas a continuacion replican los datos.

# 0 · Esquema

In [0]:
# ──── 0 · esquema ────
from pyspark.sql import functions as f

CATALOGO = "workspace"
ESQUEMA = "oceanwatch"
DELTA_POS = "/Volumes/workspace/default/oceanwatch/testing/delta_tabla"
WPI_PATH = "/Volumes/workspace/default/oceanwatch/raw/csv/UpdatedPub150.csv"

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}
    COMMENT 'OceanWatch Analytics. AIS NOAA, 1 al 7 de junio de 2023.'
""")

print(f"Esquema: {CATALOGO}.{ESQUEMA}")

Esquema: workspace.oceanwatch


# 1 · Posiciones

Tabla para el dataset AIS

In [0]:
# ──── 1 · tabla managed (UC no acepta LOCATION en un Volume) ────
(
    spark.read.format("delta").load(DELTA_POS)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.ais_posiciones")
)

spark.sql(f"""
    COMMENT ON TABLE {CATALOGO}.{ESQUEMA}.ais_posiciones IS
    'Posiciones AIS de la semana. Una fila por mensaje. Layout: CLUSTER BY (fecha, LAT, LON) para la consulta del operador por día y zona.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOGO}.{ESQUEMA}.ais_posiciones
    CLUSTER BY (fecha, LAT, LON)
""")

columnas_pos = {
    "MMSI": "Identificador del transmisor AIS. 9 dígitos (ITU-R M.585).",
    "BaseDateTime": "Instante UTC del mensaje.",
    "LAT": "Latitud en grados. 91 = no disponible (USCG / ITU-R M.1371).",
    "LON": "Longitud en grados. 181 = no disponible.",
    "SOG": "Velocidad sobre el fondo, nudos. 102.3 = no disponible.",
    "COG": "Rumbo sobre el fondo, grados. 360 = no disponible.",
    "Heading": "Dirección de la proa, grados. 511 = no disponible.",
    "VesselName": "Nombre reportado por el buque. Puede venir vacío.",
    "IMO": "Número IMO del casco, si lo tiene.",
    "CallSign": "Distintivo de llamada.",
    "VesselType": "Código de tipo de buque AIS.",
    "Status": "Estado de navegación de ese mensaje (1 fondeado, 5 amarrado).",
    "Length": "Eslora reportada, metros.",
    "Width": "Manga reportada, metros.",
    "Draft": "Calado reportado, metros.",
    "Cargo": "Código de carga AIS.",
    "TransceiverClass": "Clase del transceptor: A o B.",
    "fecha": "Día del mensaje. Llave del clustering.",
}

for col, texto in columnas_pos.items():
    spark.sql(f"""
        ALTER TABLE {CATALOGO}.{ESQUEMA}.ais_posiciones
        ALTER COLUMN {col} COMMENT '{texto}'
    """)

print("ais_posiciones registrada")

ais_posiciones registrada


# 2 · Puertos

Tabla para el World Port Index, utilizando el .csv previamente almacenado

In [0]:
# ──── 2 · puertos ────
puertos = (
    spark.read.option("header", True).option("inferSchema", True).csv(WPI_PATH)
    .select(
        f.col("Main Port Name").alias("puerto"),
        f.col("Country Code").alias("pais"),
        f.col("Latitude").cast("double").alias("lat"),
        f.col("Longitude").cast("double").alias("lon"),
    )
    .filter(f.col("lat").isNotNull() & f.col("lon").isNotNull())
    .withColumn("h3", f.expr("h3_longlatash3(lon, lat, 8)"))
)

(
    puertos.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.puertos")
)

spark.sql(f"""
    COMMENT ON TABLE {CATALOGO}.{ESQUEMA}.puertos IS
    'World Port Index (Pub 150), una fila por puerto. h3 resolución 8 para cruzar con las celdas de tráfico.'
""")

for col, texto in {
    "puerto": "Nombre del puerto en el World Port Index.",
    "pais": "País o código de país del índice.",
    "lat": "Latitud del puerto, grados.",
    "lon": "Longitud del puerto, grados.",
    "h3": "Celda H3 resolución 8 del punto del puerto.",
}.items():
    spark.sql(f"""
        ALTER TABLE {CATALOGO}.{ESQUEMA}.puertos
        ALTER COLUMN {col} COMMENT '{texto}'
    """)

print(f"puertos: {spark.table(f'{CATALOGO}.{ESQUEMA}.puertos').count():,}")

puertos: 3,824


# 3 · Buques
Tabla de la informacion de los Buques con sus valores estaticos. Una fila por valor de MSI


In [0]:
# ──── 3 · buques ────
buques = (
    spark.table(f"{CATALOGO}.{ESQUEMA}.ais_posiciones")
    .groupBy("MMSI")
    .agg(
        f.max("VesselName").alias("VesselName"), # Se aplica funcion max para obtener el valor maximo, por si hay nulos o duplicados.
        f.max("IMO").alias("IMO"),
        f.max("CallSign").alias("CallSign"),
        f.max("VesselType").alias("VesselType"),
        f.max("Length").alias("Length"),
        f.max("Width").alias("Width"),
        f.max("Draft").alias("Draft"),
        f.max("TransceiverClass").alias("TransceiverClass"),
    )
)

(
    buques.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.buques")
)

spark.sql(f"""
    COMMENT ON TABLE {CATALOGO}.{ESQUEMA}.buques IS
    'Un buque por MMSI en la semana del 1 al 7 de junio de 2023. Atributos estáticos; si hubo varios valores, se guardó el max.'
""")

for col, texto in {
    "MMSI": "Identificador del transmisor. Llave de esta tabla.",
    "VesselName": "Nombre. max de los valores no nulos de la semana.",
    "IMO": "Número IMO, si apareció.",
    "CallSign": "Distintivo de llamada, si apareció.",
    "VesselType": "Código de tipo. max si el mismo MMSI reportó más de uno.",
    "Length": "Eslora en metros. max de lo reportado.",
    "Width": "Manga en metros.",
    "Draft": "Calado en metros.",
    "TransceiverClass": "Clase A o B.",
}.items():
    spark.sql(f"""
        ALTER TABLE {CATALOGO}.{ESQUEMA}.buques
        ALTER COLUMN {col} COMMENT '{texto}'
    """)

print(f"buques: {spark.table(f'{CATALOGO}.{ESQUEMA}.buques').count():,}")

buques: 35,691


# 4 · Evidencia


In [0]:
# ──── 4 · evidencia ────
for tabla in ("ais_posiciones", "puertos", "buques"):
    print("=" * 60)
    print(tabla)
    spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOGO}.{ESQUEMA}.{tabla}").show(80, truncate=False)

ais_posiciones
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------+
|col_name                    |data_type              